In [186]:
import pandas as pd

# 데이터 melt 후 합치기

In [187]:
df2023h2 = pd.read_csv("서울교통공사_지하철혼잡도정보_20231231.csv", encoding="euc-kr")
df2024h1 = pd.read_csv("서울교통공사_지하철혼잡도정보_20240331.csv", encoding="euc-kr")
df2024h2 = pd.read_csv("서울교통공사_지하철혼잡도정보_20240630.csv", encoding="euc-kr")
df2024h3 = pd.read_csv("서울교통공사_지하철혼잡도정보_20241231.csv", encoding="euc-kr")
df2025h1 = pd.read_excel("서울교통공사_지하철혼잡도정보_20250331.xlsx")

In [188]:
df2025h1.head(3)

,연번,요일구분,호선,역번호,출발역,상하구분,5시30분,6시00분,6시30분,7시00분,...,20시00분,20시30분,21시00분,21시30분,22시00분,22시30분,23시00분,23시30분,00시00분,00시30분
0,1,평일,1,158,청량리,상선,7.2,6.9,4.5,8.3,...,24.8,26.1,28.2,24.5,23.0,22.2,21.7,14.9,8.5,0.0
1,2,평일,1,157,제기동,상선,7.6,8.7,6.5,8.7,...,30.0,26.0,34.8,27.5,25.7,25.4,24.2,16.8,11.6,0.0
2,3,평일,1,156,신설동,상선,6.7,11.2,7.2,9.6,...,30.7,26.8,36.3,28.6,26.6,26.1,25.2,16.1,12.6,0.0


In [189]:
meta_cols = ['요일구분', '호선', '역번호', '출발역', '상하구분']

In [190]:
time_cols = [col for col in df2023h2.columns if '시' in col]
time_cols[:5]

['5시30분', '6시00분', '6시30분', '7시00분', '7시30분']

In [191]:
def to_long(df, meta_cols, time_cols, period_name=None):
    df_long = df.melt(
        id_vars=meta_cols,
        value_vars=time_cols,
        var_name='time_slot',
        value_name='congestion'
    )

    if period_name is not None:
        df_long['period'] = period_name

    return df_long

In [192]:
df2023h2_long = to_long(df2023h2, meta_cols, time_cols, '2023_H2')
df2024h1_long = to_long(df2024h1, meta_cols, time_cols, '2024_H1')
df2024h2_long = to_long(df2024h2, meta_cols, time_cols, '2024_H2')
df2024h3_long = to_long(df2024h3, meta_cols, time_cols, '2024_H3')
df2025h1_long = to_long(df2025h1, meta_cols, time_cols, '2025_H1')

In [193]:
def parse_time_kor(t):
    t = t.replace('시', ':').replace('분', '')
    if ':' not in t:
        t = t + ':00'
    return t.zfill(5)

df2023h2_long['time_str'] = df2023h2_long['time_slot'].apply(parse_time_kor)
df2024h1_long['time_str'] = df2024h1_long['time_slot'].apply(parse_time_kor)
df2024h2_long['time_str'] = df2024h2_long['time_slot'].apply(parse_time_kor)
df2024h3_long['time_str'] = df2024h3_long['time_slot'].apply(parse_time_kor)
df2025h1_long['time_str'] = df2025h1_long['time_slot'].apply(parse_time_kor)

In [194]:
def add_time_index(df, time_col='time_str'):
    dt = pd.to_datetime(df[time_col], format='%H:%M')
    df['time_index'] = dt.dt.hour * 2 + dt.dt.minute // 30
    return df

In [195]:
df2023h2_long = add_time_index(df2023h2_long)
df2024h1_long = add_time_index(df2024h1_long)
df2024h2_long = add_time_index(df2024h2_long)
df2024h3_long = add_time_index(df2024h2_long)
df2025h1_long = add_time_index(df2025h1_long)

In [196]:
df_all = pd.concat(
    [df2023h2_long, df2024h1_long, df2024h2_long, df2024h3_long, df2025h1_long],
    axis=0,
    ignore_index=True
)

In [197]:
df_all.head(5)

,요일구분,호선,역번호,출발역,상하구분,time_slot,congestion,period,time_str,time_index
0,평일,1,158,청량리,상선,5시30분,9.1,2023_H2,05:30,11
1,평일,1,158,청량리,하선,5시30분,20.4,2023_H2,05:30,11
2,평일,1,157,제기동,상선,5시30분,9.0,2023_H2,05:30,11
3,평일,1,157,제기동,하선,5시30분,20.5,2023_H2,05:30,11
4,평일,1,156,신설동,상선,5시30분,8.1,2023_H2,05:30,11


In [198]:
df_all.loc[df_all['출발역'] == '성수E', '출발역'] = '성수'

# 환승역 처리

In [199]:
station_list = df_all[df_all['호선'] == 8]['출발역'].unique().tolist()

print(station_list)

['암사', '천호', '강동구청', '몽촌토성', '잠실', '석촌', '송파', '가락시장', '문정', '장지', '복정', '남위례', '산성', '남한산성입구', '단대오거리', '신흥', '수진', '모란', '암사역사공원']


In [200]:
transfer_stations = [
    "청량리", "신설동", "동묘앞", "동대문", "종로5가", '종로3가', '시청', '서울역',
    "건대입구", "성수", "왕십리", "신설동", "신당", "동대문역사문화공원", "을지로4가", "을지로3가", "시청", "충정로", "홍대입구", "합정", "당산", "영등포구청", "신도림", "까치산", "대림", "신림", "사당", "교대", "강남", "선릉", "종합운동장", "잠실",
    "연신내", "불광", "종로3가", "을지로3가", "충무로", "약수", "옥수", "신사", "고속터미널", "교대", "양재", "도곡", "수서", "가락시장", "오금",
    "노원", "창동", "성신여대입구", "동대문", "동대문역사문화공원", "충무로", "서울역", "삼각지", "이촌", "동작", "총신대입구", "사당",
    "김포공항", "까치산", "영등포구청", "신길", "여의도", "공덕", "충정로", "종로3가", "을지로4가", "동대문역사문화공원", "청구", "왕십리", "군자", "천호", "강동(하남)", "올림픽공원(한국체대)", "오금",
    "불광", "연신내", "디지털미디어시티", "합정", "공덕", "효창공원앞", "삼각지", "약수", "신당", "동묘앞", "보문", "석계", "태릉입구", "신내",
    "도봉산", "노원", "태릉입구", "상봉", "군자", "건대입구", "강남구청", "논현", "고속터미널", "총신대입구", "보라매", "대림", "가산디지털단지", "온수",
    "천호", "잠실", "가락시장", "복정", "모란"
]

# ---------

# 가중치 조정
- 최근일수록 높은 가중치

In [201]:
recency_weight_map = {
    '2023_H2': 0.3,
    '2024_H1': 0.5,
    '2024_H2': 0.6,
    '2024_H3': 0.8,
    '2025_H1': 1.0
}

In [202]:
df_all['recency_weight'] = df_all['period'].map(recency_weight_map)
df_all.head(3)

,요일구분,호선,역번호,출발역,상하구분,time_slot,congestion,period,time_str,time_index,recency_weight
0,평일,1,158,청량리,상선,5시30분,9.1,2023_H2,05:30,11,0.3
1,평일,1,158,청량리,하선,5시30분,20.4,2023_H2,05:30,11,0.3
2,평일,1,157,제기동,상선,5시30분,9.0,2023_H2,05:30,11,0.3


In [203]:
# 호선 2 출발역 리스트
lst = df_all[df_all['호선'] == 2]['출발역'].unique().tolist()

# 가나다순 정렬
lst.sort()

print(lst)


['강남', '강변', '건대입구', '교대', '구로디지털단지', '구의', '까치산', '낙성대', '당산', '대림', '도림천', '동대문역사문화공원', '둔촌동', '뚝섬', '문래', '방배', '봉천', '사당', '삼성', '상왕십리', '서울대입구', '서초', '선릉', '성수', '시청', '신답', '신당', '신대방', '신도림', '신림', '신설동', '신정네거리', '신촌(지하)', '아현', '양천구청', '역삼', '영등포구청', '올림픽공원(한국체대)', '왕십리', '용답', '용두', '을지로3가', '을지로4가', '을지로입구', '이대', '잠실', '잠실나루', '잠실새내', '종합운동장', '충정로', '한양대', '합정', '홍대입구']


In [204]:
mask = df_all['출발역'].isin(['둔촌동', '올림픽공원(한국체대)'])
df_all.loc[mask, '호선'] = 5
df_all.loc[df_all['출발역'] == '신촌(지하)', '출발역'] = '신촌'

In [205]:
# 호선 2 출발역 리스트
lst = df_all[df_all['호선'] == 7]['출발역'].unique().tolist()

# 가나다순 정렬
lst.sort()

print(lst)


['가산디지털단지', '강남구청', '건대입구', '고속터미널', '공릉', '광명사거리', '군자', '남구로', '남성', '내방', '노원', '논현', '대림', '도봉산', '뚝섬유원지', '마들', '먹골', '면목', '반포', '보라매', '사가정', '상도', '상봉', '수락산', '숭실대입구', '신대방삼거리', '신풍', '어린이대공원', '온수', '용마산', '자양(뚝섬한강공원)', '장승배기', '장암', '중계', '중곡', '중화', '천왕', '철산', '청담', '총신대입구', '태릉입구', '하계', '학동']


In [206]:
df_all.loc[df_all['출발역'] == '뚝섬유원지', '출발역'] = '자양(뚝섬한강공원)'

In [207]:
df_all.loc[df_all['출발역'] == '미아삼거리', '출발역'] = '미아사거리'

In [209]:
df_all.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 325026 entries, 0 to 325025
Data columns (total 11 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   요일구분            325026 non-null  object 
 1   호선              325026 non-null  int64  
 2   역번호             325026 non-null  int64  
 3   출발역             325026 non-null  object 
 4   상하구분            325026 non-null  object 
 5   time_slot       325026 non-null  object 
 6   congestion      317689 non-null  float64
 7   period          325026 non-null  object 
 8   time_str        325026 non-null  object 
 9   time_index      325026 non-null  int32  
 10  recency_weight  325026 non-null  float64
dtypes: float64(2), int32(1), int64(2), object(6)
memory usage: 26.0+ MB


In [210]:
# 그룹화 기준 컬럼
group_cols = ['요일구분', '호선', '역번호', '출발역', '상하구분', 'time_slot', 'time_str']

# recency_weight를 적용한 가중 평균 계산
def weighted_avg(group):
    weighted_sum = (group['congestion'] * group['recency_weight']).sum()
    weight_sum = group['recency_weight'].sum()
    return weighted_sum / weight_sum if weight_sum > 0 else 0

# groupby 후 가중 평균 계산
result = df_all.groupby(group_cols).apply(weighted_avg).reset_index()
result.columns = group_cols + ['weighted_congestion']

# 또는 agg를 사용한 방법
result = df_all.groupby(group_cols).apply(
    lambda x: pd.Series({
        'weighted_congestion': (x['congestion'] * x['recency_weight']).sum() / x['recency_weight'].sum(),
        'record_count': len(x)
    })
).reset_index()

print(result.head(20))
print(f"\n총 {len(result)}개 그룹")

/tmp/ipython-input-1242897395.py:11: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  result = df_all.groupby(group_cols).apply(weighted_avg).reset_index()


   요일구분  호선  역번호  출발역 상하구분 time_slot time_str  weighted_congestion  \
0   일요일   1  150  서울역   상선    00시00분    00:00             5.566667   
1   일요일   1  150  서울역   상선    00시30분    00:30             0.000000   
2   일요일   1  150  서울역   상선    10시00분    10:00            34.706667   
3   일요일   1  150  서울역   상선    10시30분    10:30            31.140000   
4   일요일   1  150  서울역   상선    11시00분    11:00            31.663333   
5   일요일   1  150  서울역   상선    11시30분    11:30            36.523333   
6   일요일   1  150  서울역   상선    12시00분    12:00            33.583333   
7   일요일   1  150  서울역   상선    12시30분    12:30            33.200000   
8   일요일   1  150  서울역   상선    13시00분    13:00            34.983333   
9   일요일   1  150  서울역   상선    13시30분    13:30            35.126667   
10  일요일   1  150  서울역   상선    14시00분    14:00            32.660000   
11  일요일   1  150  서울역   상선    14시30분    14:30            26.440000   
12  일요일   1  150  서울역   상선    15시00분    15:00            29.370000   
13  일요일   1  150  서울

/tmp/ipython-input-1242897395.py:15: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  result = df_all.groupby(group_cols).apply(


In [214]:
result.head(5)

,요일구분,호선,역번호,출발역,상하구분,time_slot,time_str,weighted_congestion,record_count
0,일요일,1,150,서울역,상선,00시00분,00:00,5.566667,5.0
1,일요일,1,150,서울역,상선,00시30분,00:30,0.000000,5.0
2,일요일,1,150,서울역,상선,10시00분,10:00,34.706667,5.0
3,일요일,1,150,서울역,상선,10시30분,10:30,31.140000,5.0
4,일요일,1,150,서울역,상선,11시00분,11:00,31.663333,5.0


In [216]:
result = result.drop(columns=['record_count', 'time_slot'])
display(result.head())

,요일구분,호선,역번호,출발역,상하구분,time_str,weighted_congestion
0,일요일,1,150,서울역,상선,00:00,5.566667
1,일요일,1,150,서울역,상선,00:30,0.000000
2,일요일,1,150,서울역,상선,10:00,34.706667
3,일요일,1,150,서울역,상선,10:30,31.140000
4,일요일,1,150,서울역,상선,11:00,31.663333


In [218]:
result['is_transfer_station'] = result['출발역'].isin(transfer_stations)
display(result.head())

,요일구분,호선,역번호,출발역,상하구분,time_str,weighted_congestion,is_transfer_station
0,일요일,1,150,서울역,상선,00:00,5.566667,True
1,일요일,1,150,서울역,상선,00:30,0.000000,True
2,일요일,1,150,서울역,상선,10:00,34.706667,True
3,일요일,1,150,서울역,상선,10:30,31.140000,True
4,일요일,1,150,서울역,상선,11:00,31.663333,True


In [219]:
result.to_csv('result.csv', index=False, encoding='utf-8')